# E1156285 — XGBoost time-feature review

This focused companion reproduces the new Section 23–25 cells from the main working journal. Run it from the Kaggle project folder using the project's `.venv` kernel. It compares the current and filtered XGBoost models, then creates the recommended next CSV. Existing submissions are preserved.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data Set"
assert DATA_DIR.is_dir(), "Open this notebook from the Kaggle project folder."
RANDOM_SEED = 42
FEATURE_COLUMNS = [f"feat_{i}" for i in range(1, 166)]
train = pd.read_csv(DATA_DIR / "Train Data.csv")
test = pd.read_csv(DATA_DIR / "Test Data.csv")
edges = pd.read_csv(DATA_DIR / "TXS Edgelist.csv")
sample_submission = pd.read_csv(DATA_DIR / "Sample Submission" / "BT4012 Competition Submission.csv")
labels = pd.to_numeric(train["label"], errors="coerce")
assert labels.dropna().isin([0, 1]).all()
labeled = train.loc[labels.notna()].copy()
labeled["label"] = labels.loc[labels.notna()].astype(int)
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(test["txId"])
assert train["time_step"].max() < test["time_step"].min()
assert np.isfinite(labeled[FEATURE_COLUMNS].to_numpy()).all()
assert np.isfinite(test[FEATURE_COLUMNS].to_numpy()).all()
print("Known training labels:", len(labeled))
print("Test transactions:", len(test))


Known training labels: 31235
Test transactions: 15329


## 23. Review after the XGBoost public score of 0.94265

XGBoost is now our **best submitted public result: 0.94265 ROC-AUC**. The earlier Histogram Gradient Boosting ensemble achieved 0.94199. LightGBM has not been submitted. We are still below the reported 0.96 baseline.

### What did the review find?

The code uses the correct labels, trains on earlier periods, and matches probabilities to the correct submission identifiers. We found no obvious implementation mistake explaining the gap between local and public scores.

Some local checks are much easier than others. When learning stops at step 21, performance drops within particular later periods, especially steps 24, 29, 31 and 32. Learning through step 28 helps with several of those periods. This suggests that newer examples teach the model patterns it had not learned earlier. Changing a decision threshold cannot fix poor ROC-AUC ranking.

Two other findings guide the next experiment:

- **Some features closely track time.** Six numeric columns move almost in step with the time-step number. Their exact meanings are undisclosed. They may encourage the model to remember patterns from particular periods, which may transfer poorly to future periods.
- **Neighbour features are less available in the test set.** Both endpoint feature rows are supplied for 100% of edges touching training transactions, but only about 34.4% of edges touching test transactions. A graph model evaluated with all unknown validation-node features therefore faces easier feature availability than actual test prediction. The complete supplied edge list remains available; the missing information concerns neighbouring feature rows.

These observations suggest possible problems, rather than proving that any single feature causes the public-score gap.

### What else did we check?

We tested directed graph structure, directed neighbour summaries, probability smoothing, a more restrained CatBoost model, and XGBoost with 60 training-selected features. None gave a useful overall improvement over the current XGBoost on the two longer checks. The graph-summary and smoothing improvements on the easy recent check did not carry over to the harder check.

We will test one focused change: remove features that strongly track time, while retaining the existing XGBoost settings.


## 24. Candidate: XGBoost with strongly time-related features removed

### What are we changing?

Imagine that a model learns to associate a particular month with fraud, instead of learning what makes a transaction suspicious. That association may work on familiar months and fail when the calendar moves forward.

Our data does not contain month names, but some numeric features behave similarly: their values closely follow the transaction period. We test whether excluding these columns helps XGBoost rely more on other transaction patterns.

This is **feature selection**, meaning we choose which clues to give the model. We keep the same XGBoost architecture: 127 numeric bins, at most 15 leaves per tree, learning rate 0.05, L2 penalty 10, and natural class weights. We change the inputs rather than add another large parameter search.

### How do we decide which columns to remove?

We use **Spearman correlation**, a measure of whether two quantities rise or fall in a similar order. Values near +1 mean they rise together; values near −1 mean one rises while the other falls. A value near zero indicates little monotonic relationship.

The fixed rule is to remove a feature when its **absolute correlation with time is above 0.9**. This is our experimental rule, not an assignment requirement. Strong correlation does not prove that the feature is useless or reveal its actual meaning.

For every check, we calculate correlations using **only the initial fitting portion of the training data**. We never use outer validation answers or test values to select columns. The rule therefore works even when future answers are unavailable.

### How will we check the change?

We compare the original XGBoost and the filtered version on exactly the same later transactions:

| Learn from | Select features using | Internal stopping check | Refit using | Validate on |
|---|---|---|---|---|
| Steps 1–14 | Steps 1–11 | Steps 12–14 | Steps 1–14 | Steps 15–28 |
| Steps 1–21 | Steps 1–18 | Steps 19–21 | Steps 1–21 | Steps 22–35 |
| Steps 1–28 | Steps 1–25 | Steps 26–28 | Steps 1–28 | Steps 29–35 |

The internal check chooses how long to train: up to 400 rounds, stopping after 60 rounds without an AUC improvement. Then we train a fresh model on all known labels in that training period, using the chosen number of rounds. The later validation period is used only after fitting.

These periods have already influenced earlier model choices and overlap. They are development evidence, not three independent final exams. We cannot promise a future score from them.

### What will the next code cell show?

It shows which columns were removed, how many trees were selected, and each model's ROC-AUC. It also reports precision, recall and F1 at 0.5 for explanation. The comparison and eventual submission use the original probability scores.


In [2]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from threadpoolctl import threadpool_limits

TIME_FILTER_CHECKS = [(14, 15, 28), (21, 22, 35), (28, 29, 35)]
TIME_CORRELATION_LIMIT = 0.9

def make_time_review_xgboost(rounds, *, stopping=False):
    return XGBClassifier(
        n_estimators=int(rounds), learning_rate=0.05, tree_method="hist",
        grow_policy="lossguide", max_depth=0, max_leaves=15, max_bin=127,
        min_child_weight=3, reg_lambda=10, subsample=1.0,
        colsample_bytree=1.0, scale_pos_weight=1.0,
        objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=60 if stopping else None,
        n_jobs=4, random_state=RANDOM_SEED,
    )

def select_time_review_features(learning, *, filter_time_features):
    training_end = int(learning["time_step"].max())
    fitting = learning.loc[learning["time_step"] <= training_end - 3]
    correlations = fitting[FEATURE_COLUMNS].corrwith(
        fitting["time_step"], method="spearman"
    )
    removed = (
        correlations.index[correlations.abs() > TIME_CORRELATION_LIMIT].tolist()
        if filter_time_features else []
    )
    columns = [column for column in FEATURE_COLUMNS if column not in removed]
    assert columns and "time_step" not in columns and "label" not in columns
    return columns, removed, correlations

def fit_time_review_model(learning, *, filter_time_features):
    training_end = int(learning["time_step"].max())
    fitting = learning.loc[learning["time_step"] <= training_end - 3]
    stopping = learning.loc[learning["time_step"] > training_end - 3]
    assert fitting["time_step"].max() < stopping["time_step"].min()
    assert fitting["label"].nunique() == stopping["label"].nunique() == 2
    columns, removed, correlations = select_time_review_features(
        learning, filter_time_features=filter_time_features
    )
    with threadpool_limits(limits=4):
        initial = make_time_review_xgboost(400, stopping=True)
        initial.fit(
            fitting[columns], fitting["label"],
            eval_set=[(stopping[columns], stopping["label"])], verbose=False,
        )
        rounds = int(initial.best_iteration + 1)
        assert 1 <= rounds <= 400
        refitted = make_time_review_xgboost(rounds)
        refitted.fit(learning[columns], learning["label"])
    return refitted, columns, removed, rounds, correlations

time_review_records = []
time_review_predictions = {}
for training_end, validation_start, validation_end in TIME_FILTER_CHECKS:
    learning = labeled.loc[labeled["time_step"] <= training_end]
    checking = labeled.loc[labeled["time_step"].between(validation_start, validation_end)]
    assert learning["time_step"].max() < checking["time_step"].min()
    assert set(learning["txId"]).isdisjoint(checking["txId"])
    for display_name, filtering in [("Original XGBoost", False), ("Time-filtered XGBoost", True)]:
        fitted, columns, removed, rounds, _ = fit_time_review_model(
            learning, filter_time_features=filtering
        )
        with threadpool_limits(limits=4):
            illicit_column = list(fitted.classes_).index(1)
            probabilities = fitted.predict_proba(checking[columns])[:, illicit_column]
        time_review_predictions[(display_name, training_end)] = probabilities
        decisions = (probabilities >= 0.5).astype(int)
        time_review_records.append({
            "model": display_name, "train_end": training_end,
            "validation_steps": f"{validation_start}–{validation_end}",
            "features": len(columns), "rounds": rounds,
            "validation_auc": roc_auc_score(checking["label"], probabilities),
            "precision_at_0.5": precision_score(checking["label"], decisions, zero_division=0),
            "recall_at_0.5": recall_score(checking["label"], decisions, zero_division=0),
            "f1_at_0.5": f1_score(checking["label"], decisions, zero_division=0),
            "removed_features": ", ".join(removed),
        })
        print(f"{display_name}: train 1–{training_end}; validate {validation_start}–{validation_end}; "
              f"AUC {time_review_records[-1]['validation_auc']:.6f}; removed {removed}", flush=True)

time_review_results = pd.DataFrame(time_review_records)
display(time_review_results.round(4))
time_review_auc = time_review_results.pivot(index="model", columns="validation_steps", values="validation_auc")
display(time_review_auc.round(6))
time_review_long_mean = time_review_results.loc[time_review_results["train_end"].isin([14, 21])].groupby("model")["validation_auc"].mean()
print("Equal-weight average of the two fourteen-step checks:")
display(time_review_long_mean.round(6))


Original XGBoost: train 1–14; validate 15–28; AUC 0.968521; removed []


Time-filtered XGBoost: train 1–14; validate 15–28; AUC 0.967511; removed ['feat_101', 'feat_103', 'feat_136', 'feat_139']


Original XGBoost: train 1–21; validate 22–35; AUC 0.915330; removed []


Time-filtered XGBoost: train 1–21; validate 22–35; AUC 0.921428; removed ['feat_100', 'feat_101', 'feat_103', 'feat_136', 'feat_137', 'feat_139']


Original XGBoost: train 1–28; validate 29–35; AUC 0.995816; removed []


Time-filtered XGBoost: train 1–28; validate 29–35; AUC 0.995455; removed ['feat_100', 'feat_101', 'feat_103', 'feat_136', 'feat_137', 'feat_139']


,model,train_end,validation_steps,features,rounds,validation_auc,precision_at_0.5,recall_at_0.5,f1_at_0.5,removed_features
0,Original XGBoost,14,15–28,165,185,0.9685,0.9770,0.8022,0.8810,
1,Time-filtered XGBoost,14,15–28,161,186,0.9675,0.9785,0.8003,0.8805,"feat_101, feat_103, feat_136, feat_139"
2,Original XGBoost,21,22–35,165,381,0.9153,0.9291,0.6723,0.7801,
3,Time-filtered XGBoost,21,22–35,159,315,0.9214,0.9517,0.6667,0.7841,"feat_100, feat_101, feat_103, feat_136, feat_1..."
4,Original XGBoost,28,29–35,165,398,0.9958,0.9770,0.8485,0.9082,
5,Time-filtered XGBoost,28,29–35,159,388,0.9955,0.9747,0.8385,0.9015,"feat_100, feat_101, feat_103, feat_136, feat_1..."


validation_steps,15–28,22–35,29–35
model,,,
Original XGBoost,0.968521,0.915330,0.995816
Time-filtered XGBoost,0.967511,0.921428,0.995455


Equal-weight average of the two fourteen-step checks:


model
Original XGBoost         0.941926
Time-filtered XGBoost    0.944470
Name: validation_auc, dtype: float64

### 24.1. What do these results mean?

| Model | Steps 15–28 | Steps 22–35 | Steps 29–35 | Mean of the two fourteen-step checks |
|---|---:|---:|---:|---:|
| Original XGBoost | 0.968521 | 0.915330 | 0.995816 | 0.941926 |
| Time-filtered XGBoost | 0.967511 | **0.921428** | 0.995455 | **0.944470** |

The change improved the harder fourteen-step check by about **0.0061 AUC**. Its average over the two longer checks rose by about **0.0025**. The earlier and short recent checks declined slightly. This supports a modest candidate improvement rather than a claim that every period improves.

We also checked an equal average of the original and filtered models. Its two-long-check mean was 0.943397, below the filtered model's 0.944470. We therefore use the filtered model alone for this candidate.

This version is our recommended **next submission candidate based on this review**. The original XGBoost remains our best **confirmed public result**, at 0.94265, until the new file is scored. Neither the public result nor the local checks have established performance above 0.96.


### 24.2. Train the final candidate and make its submission file

Now that the comparison is complete, we use all **31,235 transactions with known labels from steps 1–35**. Unknown labels are still excluded.

For the final fit:

1. Use steps 1–32 to calculate time correlations and select features.
2. Initially train on steps 1–32 and monitor steps 33–35 to choose the number of rounds.
3. Start fresh and train on all known labels from steps 1–35 using those rounds and selected features.
4. Predict an illicit probability for each of the 15,329 test transactions from steps 36–49.
5. Match probabilities to the submission template's identifiers and save a new CSV.

This removes `feat_100`, `feat_101`, `feat_103`, `feat_136`, `feat_137` and `feat_139`, leaving **159 features**. The code applies the selection rule itself rather than relying on a manually typed exclusion list.

The output is **`submission_xgboost_time_filtered.csv`**. Its `target` column contains probability scores, rather than 0/1 decisions. No 0.5 threshold is applied to the submission.

We save the trained model and its feature list so that we can reproduce the file. We also load the saved model again and verify its probabilities. Creating this file locally does not consume a Kaggle submission; uploading it does.


In [3]:
import hashlib
import json
import xgboost

time_filtered_final_model, time_filtered_columns, time_filtered_removed, time_filtered_rounds, time_filtered_correlations = fit_time_review_model(
    labeled, filter_time_features=True
)
with threadpool_limits(limits=4):
    illicit_column = list(time_filtered_final_model.classes_).index(1)
    time_filtered_probabilities = time_filtered_final_model.predict_proba(
        test[time_filtered_columns]
    )[:, illicit_column]

assert test["index"].is_unique and sample_submission["index"].is_unique
assert len(test) == len(sample_submission) == 15329
assert set(test["index"]) == set(sample_submission["index"])
probabilities_by_identifier = pd.Series(time_filtered_probabilities, index=test["index"])
time_filtered_submission = sample_submission[["index"]].copy()
time_filtered_submission["target"] = time_filtered_submission["index"].map(probabilities_by_identifier)
assert time_filtered_submission.columns.tolist() == ["index", "target"]
assert time_filtered_submission["index"].equals(sample_submission["index"])
assert np.isfinite(time_filtered_submission["target"].to_numpy()).all()
assert time_filtered_submission["target"].between(0, 1).all()

review_submission_dir = PROJECT_DIR / "submissions"
review_model_dir = PROJECT_DIR / "models"
review_submission_dir.mkdir(exist_ok=True)
review_model_dir.mkdir(exist_ok=True)
time_filtered_submission_path = review_submission_dir / "submission_xgboost_time_filtered.csv"
time_filtered_model_path = review_model_dir / "xgboost_time_filtered.json"
time_filtered_submission.to_csv(time_filtered_submission_path, index=False, float_format="%.17g")
time_filtered_final_model.save_model(time_filtered_model_path)
time_review_results.to_csv(review_submission_dir / "xgboost_time_filtered_validation.csv", index=False)
time_filtered_correlations.rename("spearman_with_time").to_csv(
    review_model_dir / "xgboost_time_filtered_correlations.csv", index_label="feature"
)

# Reload the persisted model and check the saved file against its predictions.
reloaded_time_filtered = XGBClassifier(n_jobs=4)
reloaded_time_filtered.load_model(time_filtered_model_path)
assert reloaded_time_filtered.get_booster().feature_names == time_filtered_columns
with threadpool_limits(limits=4):
    restored_probabilities = reloaded_time_filtered.predict_proba(test[time_filtered_columns])[:, 1]
assert np.array_equal(restored_probabilities, time_filtered_probabilities)
saved_submission = pd.read_csv(time_filtered_submission_path)
assert saved_submission["index"].equals(sample_submission["index"])
assert np.allclose(saved_submission["target"], time_filtered_submission["target"], rtol=0, atol=1e-12)

time_filtered_metadata = {
    "candidate": "XGBoost with training-only time-feature filtering",
    "public_score": None,
    "previous_best_public_score": 0.94265,
    "training_steps": [1, 35], "selection_steps": [1, 32],
    "internal_stopping_steps": [33, 35], "prediction_steps": [36, 49],
    "training_examples": len(labeled), "submission_rows": len(saved_submission),
    "feature_selection": "absolute Spearman correlation with time_step > 0.9 on initial fitting rows",
    "retained_features": time_filtered_columns, "removed_features": time_filtered_removed,
    "training_rounds": time_filtered_rounds, "random_seed": RANDOM_SEED,
    "model_parameters": {
        key: (None if isinstance(value, float) and not np.isfinite(value) else value)
        for key, value in time_filtered_final_model.get_params().items()
    },
    "xgboost_version": xgboost.__version__, "pandas_version": pd.__version__, "numpy_version": np.__version__,
    "mean_long_validation_auc": float(time_review_long_mean.loc["Time-filtered XGBoost"]),
    "validation_periods_reused_for_development": True,
    "submission_sha256": hashlib.sha256(time_filtered_submission_path.read_bytes()).hexdigest(),
}
(review_model_dir / "xgboost_time_filtered_metadata.json").write_text(json.dumps(time_filtered_metadata, indent=2, allow_nan=False))
display(time_filtered_submission.head())
print("Removed features:", time_filtered_removed)
print("Retained features:", len(time_filtered_columns))
print("Final training rounds:", time_filtered_rounds)
print("Submission rows:", len(saved_submission))
print("Saved to:", time_filtered_submission_path)
print("Saved-model reload and CSV checks passed.")


,index,target
0,0,0.001234
1,1,0.004682
2,2,0.003808
3,3,0.001110
4,4,0.000463


Removed features: ['feat_100', 'feat_101', 'feat_103', 'feat_136', 'feat_137', 'feat_139']
Retained features: 159
Final training rounds: 305
Submission rows: 15329
Saved to: /Users/mlvlo/BT4012/Kaggle/submissions/submission_xgboost_time_filtered.csv
Saved-model reload and CSV checks passed.


## 25. Submission decision after this review

**Upload `submission_xgboost_time_filtered.csv` next.** The reason is its stronger mean on the two longer chronological checks and the improvement in the harder period. It is a measured experiment, with a specific explanation for the changed inputs, rather than a promised route to 0.96.

| Submission | Public ROC-AUC |
|---|---:|
| Original Histogram Gradient Boosting | 0.93654 |
| Extra Trees | 0.91673 |
| Histogram Gradient Boosting bin ensemble | 0.94199 |
| Original tuned XGBoost | **0.94265** |
| LightGBM | Not submitted |
| Time-filtered XGBoost | Pending |

Keep the previous XGBoost CSV. A newer file should replace a better confirmed result only after we have evidence supporting that decision. The final private leaderboard remains unknown because the public score covers only 30% of the test rows.

After the candidate is scored, record its public score here. The report should explain both the harder-period gain and the slight declines in the other checks. Threshold testing remains pending and is separate from improving probability rankings.
